# Data Preparation — AASE 2026/27
**Preparação de dados de internamento neonatal para modelos de ML**

> ⚠️ Os datasets originais em `../dados_aase_202627/` são **intocáveis** — este notebook lê-os e escreve ficheiros novos em `03_data_preparation/`.

| Output | Descrição |
|---|---|
| `dataset_final.csv` | Dataset agregado por episódio (features + targets) |
| `train.csv` | 80% para treino (estratificado por patologia) |
| `test.csv` | 20% para teste |
| `metadata.csv` | Mapeamento label → patologia + contagens |
| `normalisation_params.csv` | Parâmetros z-score calculados no treino |

## 0. Imports e configuração

In [ ]:
import csv
import math
import random
import statistics
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

%matplotlib inline

# paleta idêntica ao EDA
C = {
    "blue":   "#2a78d6",
    "orange": "#eb6834",
    "teal":   "#1baf7a",
    "yellow": "#eda100",
    "red":    "#d03b3b",
    "grey":   "#8996ae",
    "bg":     "#f4f7fc",
    "surf":   "#ffffff",
    "text":   "#0d1219",
    "text2":  "#4a5568",
}

plt.rcParams.update({
    "figure.facecolor":  C["bg"],
    "axes.facecolor":    C["surf"],
    "axes.edgecolor":    "#dce3f0",
    "axes.labelcolor":   C["text2"],
    "axes.grid":         True,
    "axes.axisbelow":    True,
    "grid.color":        "#e6eaf3",
    "grid.linewidth":    0.8,
    "xtick.color":       C["text2"],
    "ytick.color":       C["text2"],
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "font.family":       "sans-serif",
    "font.size":         10,
    "axes.titlesize":    12,
    "axes.titleweight":  "semibold",
    "axes.titlepad":     10,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        110,
})

PASTA   = "../dados_aase_202627"   # dados originais — só leitura
OUT_DIR = Path(".")
SEED    = 42
random.seed(SEED)
print("Pronto.")

## 1. Leitura dos dados originais (read-only)

In [ ]:
def safe_float(v):
    try:
        return float(v)
    except (ValueError, TypeError):
        return None

def norm_ep(ep_id):
    """Normaliza IDs: remove espaços e converte para maiúsculas.
    Os dados originais têm inconsistências: ' EP011', 'EP 016', 'ep016'.
    Esta função resolve em memória sem alterar os ficheiros originais."""
    return ep_id.strip().replace(" ", "").upper()

MAPA_PAT = {
    "doenca das membranas hialinas":  "Doença das Membranas Hialinas",
    "doença das membranas hialinas":  "Doença das Membranas Hialinas",
    "dmh":                            "Doença das Membranas Hialinas",
    "icterícia neonatal":             "Icterícia Neonatal",
    "ictericia neonatal":             "Icterícia Neonatal",
    "anemia":                         "Anemia",
    "citomegalovírus":                "Citomegalovírus",
    "citomegalovirus":                "Citomegalovírus",
    "cmv":                            "Citomegalovírus",
    "displasia broncopulmonar":       "Displasia Broncopulmonar",
    "persistência do canal arterial": "Persistência do Canal Arterial",
    "refluxo gastroesofágico":        "Refluxo Gastroesofágico",
    "sépsis":                         "Sépsis",
    "lesão renal aguda":              "Lesão Renal Aguda",
    "retinopatia da prematuridade":   "Retinopatia da Prematuridade",
    "icterícia":                      "Icterícia Neonatal",
    "ictericia":                      "Icterícia Neonatal",
    "rge":                            "Refluxo Gastroesofágico",
}

def norm_pat(p):
    return MAPA_PAT.get(p.strip().lower(), p.strip().title())

# leitura — nunca se escreve nestes ficheiros
with open(f"{PASTA}/1_sinais_vitais_por_hora-1.csv", newline="", encoding="utf-8") as f:
    sv_raw = list(csv.DictReader(f))

with open(f"{PASTA}/3_patologias_por_dia-1.csv", newline="", encoding="utf-8") as f:
    pat_raw = list(csv.DictReader(f))

# normalizar IDs em memória (ficheiros originais não são alterados)
for r in sv_raw:
    r["episodio"] = norm_ep(r["episodio"])
for r in pat_raw:
    r["episodio"] = norm_ep(r["episodio"])

ep_ids_sv  = set(r["episodio"] for r in sv_raw)
ep_ids_pat = set(r["episodio"] for r in pat_raw)

print(f"Sinais vitais brutos: {len(sv_raw):,}")
print(f"Patologias brutas:    {len(pat_raw):,}")
print(f"IDs únicos SV:        {len(ep_ids_sv)}")
print(f"IDs únicos Pat:       {len(ep_ids_pat)}")
print(f"IDs em ambos:         {len(ep_ids_sv & ep_ids_pat)}")
so_sv  = ep_ids_sv  - ep_ids_pat
so_pat = ep_ids_pat - ep_ids_sv
if so_sv:  print(f"Só em SV:  {sorted(so_sv)}")
if so_pat: print(f"Só em Pat: {sorted(so_pat)}")

## 2. Limpeza — exclusão de registos inválidos

Estratégia: **exclusão** dos registos fora dos limiares fisiológicos.  
Justificação: valores fora dos intervalos (SpO₂ ∈ [50,100], FC ∈ [50,350], Perfusão ∈ [0,20]) representam erros de sensor, não estados clínicos. A imputação introduziria ruído artificial; a exclusão (2,5%) preserva a integridade clínica.

In [ ]:
sv_clean, sv_inv = [], []
for r in sv_raw:
    sat  = safe_float(r["sat_o2"])
    puls = safe_float(r["puls_rate"])
    perf = safe_float(r["perfusion"])
    hora = safe_float(r["hora"])
    ok = (
        sat  is not None and 50 <= sat  <= 100 and
        puls is not None and 50 <= puls <= 350 and
        perf is not None and 0  <= perf <= 20  and
        hora is not None and 0  <= hora <= 23
    )
    (sv_clean if ok else sv_inv).append(r)

print(f"Válidos:   {len(sv_clean):,}  ({len(sv_clean)/len(sv_raw)*100:.1f}%)")
print(f"Excluídos: {len(sv_inv):,}   ({len(sv_inv)/len(sv_raw)*100:.1f}%)")

prob = Counter()
for r in sv_inv:
    sat  = safe_float(r["sat_o2"])
    puls = safe_float(r["puls_rate"])
    perf = safe_float(r["perfusion"])
    hora = safe_float(r["hora"])
    if sat  is None or not (50 <= sat  <= 100): prob["SpO₂ fora [50,100]"]   += 1
    if puls is None or not (50 <= puls <= 350): prob["FC fora [50,350]"]     += 1
    if perf is None or not (0  <= perf <= 20):  prob["Perfusão fora [0,20]"] += 1
    if hora is None or not (0  <= hora <= 23):  prob["Hora inválida"]        += 1

print()
for k, v in prob.most_common():
    print(f"  {k}: {v}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("Limpeza de Dados", fontsize=13, fontweight="bold", x=0.05, ha="left")

ax = axes[0]
labels_q = ["Válidos", "Excluídos"]
vals_q   = [len(sv_clean), len(sv_inv)]
ax.bar(labels_q, vals_q, color=[C["teal"], C["red"]],
       width=0.45, zorder=3, edgecolor=C["surf"], linewidth=0.5)
ax.set_title("Volume após limpeza")
ax.set_ylabel("Nº registos")
for i, v in enumerate(vals_q):
    ax.text(i, v + 50, f"{v:,}", ha="center", fontsize=9, color=C["text2"])

ax = axes[1]
labels_p = list(prob.keys())
vals_p   = [prob[k] for k in labels_p]
cols_p   = [C["red"], C["orange"], C["yellow"], C["grey"]]
ax.bar(labels_p, vals_p, color=cols_p[:len(labels_p)],
       width=0.5, zorder=3, edgecolor=C["surf"], linewidth=0.5)
ax.set_title("Registos inválidos por tipo")
ax.set_ylabel("Nº registos")
ax.set_xticklabels(labels_p, rotation=15, ha="right", fontsize=8.5)
for i, v in enumerate(vals_p):
    ax.text(i, v + 1, str(v), ha="center", fontsize=9, color=C["text2"])

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 3. Agregação por episódio

Grão final: **1 linha = 1 episódio**.

| Feature | Descrição |
|---|---|
| `sat_o2_{mean,std,min,max}` | Estatísticas de SpO₂ |
| `puls_rate_{mean,std,min,max}` | Estatísticas de FC |
| `perfusion_{mean,std,min,max}` | Estatísticas de perfusão |
| `desat_count` | Nº de registos com SpO₂ < 90% |
| `desat_pct` | % de registos com dessaturação |
| `carga_dessaturacao` | Σ(90 − SpO₂) para SpO₂ < 90 (área de défice) |
| `duracao_dias` | Nº de dias distintos com sinais vitais registados |
| `n_registos` | Nº de registos válidos por episódio |

In [ ]:
ep_sv = defaultdict(lambda: {"sat": [], "puls": [], "perf": [], "dias": set()})

for r in sv_clean:
    ep = r["episodio"]
    ep_sv[ep]["sat"].append(safe_float(r["sat_o2"]))
    ep_sv[ep]["puls"].append(safe_float(r["puls_rate"]))
    ep_sv[ep]["perf"].append(safe_float(r["perfusion"]))
    ep_sv[ep]["dias"].add(r["data"])

def ep_features(ep_id):
    d = ep_sv[ep_id]
    sat, puls, perf = d["sat"], d["puls"], d["perf"]
    n = len(sat)
    desat = [v for v in sat if v < 90]
    return {
        "episodio":           ep_id,
        "n_registos":         n,
        "sat_o2_mean":        round(statistics.mean(sat), 4),
        "sat_o2_std":         round(statistics.stdev(sat) if n > 1 else 0, 4),
        "sat_o2_min":         round(min(sat), 4),
        "sat_o2_max":         round(max(sat), 4),
        "puls_rate_mean":     round(statistics.mean(puls), 4),
        "puls_rate_std":      round(statistics.stdev(puls) if n > 1 else 0, 4),
        "puls_rate_min":      round(min(puls), 4),
        "puls_rate_max":      round(max(puls), 4),
        "perfusion_mean":     round(statistics.mean(perf), 4),
        "perfusion_std":      round(statistics.stdev(perf) if n > 1 else 0, 4),
        "perfusion_min":      round(min(perf), 4),
        "perfusion_max":      round(max(perf), 4),
        "desat_count":        len(desat),
        "desat_pct":          round(len(desat) / n * 100, 4),
        "carga_dessaturacao": round(sum(90 - v for v in desat), 4),
        "duracao_dias":       len(d["dias"]),
    }

eps_sorted = sorted(ep_sv.keys())
features   = [ep_features(ep) for ep in eps_sorted]

print(f"Episódios com sinais vitais: {len(features)}")
print(f"\nExemplo — {features[0]['episodio']}:")
for k, v in features[0].items():
    print(f"  {k:<24} {v}")

## 4. Targets

### 4a. Alvo de classificação — patologia predominante
Para cada episódio: a patologia com mais registos dia×episódio.  
Em empate usa-se a primeira por ordem alfabética (determinístico).

### 4b. Alvo de regressão
Dois candidatos (ambos incluídos; escolha final confirmada com docente):
- `duracao_dias` — dias distintos com sinais vitais registados
- `carga_dessaturacao` — Σ(90 − SpO₂) para SpO₂ < 90%

In [ ]:
ep_pat_count = defaultdict(Counter)
for r in pat_raw:
    if r["patologia"].strip():
        ep_pat_count[r["episodio"]][norm_pat(r["patologia"])] += 1

def pat_predominante(ep_id):
    if ep_id not in ep_pat_count or not ep_pat_count[ep_id]:
        return None
    cnt = ep_pat_count[ep_id]
    max_n = max(cnt.values())
    return sorted(p for p, n in cnt.items() if n == max_n)[0]

sem_pat = []
for row in features:
    pp = pat_predominante(row["episodio"])
    row["pat_predominante"] = pp
    if pp is None:
        sem_pat.append(row["episodio"])

dist_pat = Counter(r["pat_predominante"] for r in features if r["pat_predominante"])

print(f"Com patologia: {sum(1 for r in features if r['pat_predominante'])}/{len(features)}")
if sem_pat:
    print(f"Sem patologia (serão excluídos do split): {sem_pat}")
print()
for p, n in dist_pat.most_common():
    print(f"  {n:3d}  {p}")

In [ ]:
duracoes = [r["duracao_dias"]       for r in features]
cargas   = [r["carga_dessaturacao"] for r in features]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle("Distribuição dos Targets", fontsize=13, fontweight="bold",
             x=0.05, ha="left")

ax = axes[0]
ax.hist(duracoes, bins=range(1, max(duracoes) + 2), color=C["blue"],
        edgecolor=C["surf"], linewidth=0.5, zorder=3, align="left")
ax.axvline(statistics.mean(duracoes), color=C["orange"], lw=1.5, ls="--",
           label=f"Média {statistics.mean(duracoes):.1f} dias")
ax.axvline(statistics.median(duracoes), color=C["teal"], lw=1.5, ls="-.",
           label=f"Mediana {statistics.median(duracoes):.0f} dias")
ax.set_title("Duração do Internamento")
ax.set_xlabel("Dias")
ax.set_ylabel("Nº episódios")
ax.legend(fontsize=8.5, frameon=False)

ax = axes[1]
ax.hist(cargas, bins=20, color=C["orange"],
        edgecolor=C["surf"], linewidth=0.5, zorder=3)
ax.axvline(statistics.mean(cargas), color=C["blue"], lw=1.5, ls="--",
           label=f"Média {statistics.mean(cargas):.1f}")
ax.set_title("Carga de Dessaturação")
ax.set_xlabel("Σ(90 − SpO₂) para SpO₂ < 90%")
ax.set_ylabel("Nº episódios")
ax.legend(fontsize=8.5, frameon=False)

ax = axes[2]
pats_ord = [p for p, _ in dist_pat.most_common()]
counts_p = [dist_pat[p] for p in pats_ord]
cols_loop = [C["blue"], C["orange"], C["teal"], C["yellow"], C["red"], C["grey"]]
ax.barh(range(len(pats_ord)), counts_p,
        color=[cols_loop[i % len(cols_loop)] for i in range(len(pats_ord))],
        height=0.65, zorder=3, edgecolor=C["surf"], linewidth=0.5)
ax.set_yticks(range(len(pats_ord)))
ax.set_yticklabels(pats_ord, fontsize=8)
ax.invert_yaxis()
ax.set_title("Patologia Predominante")
ax.set_xlabel("Nº episódios")
ax.grid(axis="y", visible=False)
for i, v in enumerate(counts_p):
    ax.text(v + 0.1, i, str(v), va="center", fontsize=8, color=C["text2"])

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 5. Codificação e normalização

### 5a. Label encoding
Atribui inteiro 0–N a cada patologia (ordem de frequência decrescente).  
Episódios sem patologia recebem `pat_label = -1` e são excluídos do split mas mantidos no `dataset_final.csv`.

### 5b. Normalização Z-score
Os parâmetros (média, DP) são calculados **apenas no conjunto de treino** para evitar data leakage.  
O mesmo escalonamento é aplicado ao teste.

In [ ]:
pats_por_freq = [p for p, _ in dist_pat.most_common()]
pat_to_label  = {p: i for i, p in enumerate(pats_por_freq)}
label_to_pat  = {i: p for p, i in pat_to_label.items()}

print("Mapeamento patologia → label:")
for p, lbl in pat_to_label.items():
    print(f"  {lbl:2d}  {p}  (n={dist_pat[p]})")

for row in features:
    if row["pat_predominante"]:
        row["pat_label"] = pat_to_label[row["pat_predominante"]]
    else:
        row["pat_label"] = -1

FEATURE_COLS = [
    "sat_o2_mean",    "sat_o2_std",    "sat_o2_min",    "sat_o2_max",
    "puls_rate_mean", "puls_rate_std", "puls_rate_min", "puls_rate_max",
    "perfusion_mean", "perfusion_std", "perfusion_min", "perfusion_max",
    "desat_count",    "desat_pct",     "carga_dessaturacao",
    "n_registos",
]

def z_score_params(rows, cols):
    params = {}
    for c in cols:
        vals = [r[c] for r in rows]
        mu   = statistics.mean(vals)
        sd   = statistics.stdev(vals) if len(vals) > 1 else 1.0
        params[c] = (mu, max(sd, 1e-9))
    return params

def apply_z_score(rows, params):
    for row in rows:
        for c, (mu, sd) in params.items():
            row[c + "_z"] = round((row[c] - mu) / sd, 6)

print(f"\nFeatures numéricas: {len(FEATURE_COLS)}")

## 6. Divisão treino/teste — 80/20 estratificado

Estratificação por `pat_label`. Episódios sem patologia (`pat_label == -1`) são mantidos no `dataset_final.csv` mas excluídos do split.

In [ ]:
features_com_pat = [r for r in features if r["pat_label"] >= 0]

por_classe = defaultdict(list)
for row in features_com_pat:
    por_classe[row["pat_label"]].append(row)

train_rows, test_rows = [], []
for lbl, rows in sorted(por_classe.items()):
    shuffled = rows[:]
    random.shuffle(shuffled)
    n_train = max(1, round(len(shuffled) * 0.8))
    train_rows.extend(shuffled[:n_train])
    test_rows.extend(shuffled[n_train:])

# parâmetros de normalização calculados APENAS no treino
params_train = z_score_params(train_rows, FEATURE_COLS)
apply_z_score(train_rows, params_train)
apply_z_score(test_rows,  params_train)

print(f"Episódios com patologia: {len(features_com_pat)}")
print(f"Treino: {len(train_rows)}  |  Teste: {len(test_rows)}")
print()

train_dist = Counter(r["pat_predominante"] for r in train_rows)
test_dist  = Counter(r["pat_predominante"] for r in test_rows)

print(f"{'Patologia':<36}  {'Treino':>6}  {'Teste':>6}  {'% treino':>8}")
print("-" * 62)
for p in pats_por_freq:
    tr  = train_dist.get(p, 0)
    te  = test_dist.get(p, 0)
    tot = tr + te
    pct = tr / tot * 100 if tot else 0
    print(f"  {p:<34}  {tr:>6}  {te:>6}  {pct:>7.1f}%")

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(pats_por_freq))
w = 0.38
ax.bar(x - w/2, [train_dist.get(p,0) for p in pats_por_freq],
       width=w, label="Treino", color=C["blue"], zorder=3,
       edgecolor=C["surf"], linewidth=0.5)
ax.bar(x + w/2, [test_dist.get(p,0)  for p in pats_por_freq],
       width=w, label="Teste",  color=C["orange"], zorder=3,
       edgecolor=C["surf"], linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(pats_por_freq, rotation=35, ha="right", fontsize=8.5)
ax.set_title("Distribuição de Classes — Treino vs Teste",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Nº episódios")
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

## 7. Guardar outputs

| Ficheiro | Conteúdo |
|---|---|
| `dataset_final.csv` | Todos os 58 episódios com features + targets (sem normalização) |
| `train.csv` | Conjunto de treino com features normalizadas (`_z`) |
| `test.csv` | Conjunto de teste com mesma normalização |
| `metadata.csv` | Mapeamento label → patologia + contagens |
| `normalisation_params.csv` | Parâmetros z-score (média e DP do treino) |

In [ ]:
def write_csv(path, rows, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    print(f"  {Path(path).name:<32}  {len(rows):>4} linhas")

BASE_COLS   = ["episodio", "n_registos"]
TARGET_COLS = ["duracao_dias", "carga_dessaturacao", "pat_predominante", "pat_label"]
Z_COLS      = [c + "_z" for c in FEATURE_COLS]

print("Ficheiros criados:")
write_csv(OUT_DIR / "dataset_final.csv",
          features,
          BASE_COLS + FEATURE_COLS + TARGET_COLS)

write_csv(OUT_DIR / "train.csv",
          train_rows,
          BASE_COLS + FEATURE_COLS + Z_COLS + TARGET_COLS)

write_csv(OUT_DIR / "test.csv",
          test_rows,
          BASE_COLS + FEATURE_COLS + Z_COLS + TARGET_COLS)

write_csv(OUT_DIR / "metadata.csv",
          [{"pat_label": lbl, "pat_nome": label_to_pat[lbl],
            "n_episodios": dist_pat.get(label_to_pat[lbl], 0)}
           for lbl in sorted(label_to_pat)],
          ["pat_label", "pat_nome", "n_episodios"])

write_csv(OUT_DIR / "normalisation_params.csv",
          [{"feature": c, "mean": round(mu, 6), "std": round(sd, 6)}
           for c, (mu, sd) in params_train.items()],
          ["feature", "mean", "std"])

print("\nDatasets originais não foram alterados.")

## 8. Verificação final

In [ ]:
import os

outputs = ["dataset_final.csv", "train.csv", "test.csv",
           "metadata.csv", "normalisation_params.csv"]

print("Ficheiros em 03_data_preparation/:")
print(f"  {'Ficheiro':<32}  {'Linhas':>6}  {'Colunas':>7}  {'Tamanho':>8}")
print("-" * 60)
for fname in outputs:
    path = OUT_DIR / fname
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    size_kb = os.path.getsize(path) / 1024
    print(f"  {fname:<32}  {len(rows):>6}  {len(rows[0]) if rows else 0:>7}  {size_kb:>6.1f} KB")

print("\nDatasets originais (verificação de integridade):")
for fname in ["1_sinais_vitais_por_hora-1.csv", "3_patologias_por_dia-1.csv"]:
    path = f"{PASTA}/{fname}"
    with open(path, newline="", encoding="utf-8") as f:
        n = sum(1 for _ in csv.reader(f)) - 1
    print(f"  {fname}  ({n:,} registos) ✓")

print(f"""
Resumo:
  Episódios total:          {len(features)}
  Com patologia definida:   {len(features_com_pat)}
  Treino:                   {len(train_rows)}
  Teste:                    {len(test_rows)}
  Features numéricas:       {len(FEATURE_COLS)}
  Classes de patologia:     {len(pat_to_label)}
  Registos SV excluídos:    {len(sv_inv)} (2.5%)
""")